# Pilhas e filas

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Escolher entre uma pilha e uma fila para uma tarefa, e usar ArrayDeque como qualquer uma das duas
- Simular um processo que funciona em turnos com uma fila
- Achar o próximo valor maior de cada elemento com uma pilha monotônica, em O(n)
- Manter o máximo de uma janela deslizante com um deque monotônico
- Calcular uma expressão com precedência de operadores usando duas pilhas

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Uma **pilha** devolve o último valor colocado (último a entrar, primeiro a sair): o histórico de desfazer, o botão Voltar do navegador, a conferência de parênteses, as chamadas de método de um programa rodando. Uma **fila** devolve o primeiro (primeiro a entrar, primeiro a sair): trabalhos de impressão, uma CPU revezando entre programas, mensagens esperando para ser tratadas.

Em Java, o `ArrayDeque` é as duas coisas. Como pilha: `push`, `pop` e `peek`. Como fila: `offer`, `poll` e `peek`. Essas operações nas pontas levam tempo O(1) amortizado.

![Uma pilha e uma fila num ArrayDeque](deque.pt-br.svg)

**Um ArrayDeque, usado dos dois jeitos:**

In [ ]:
Deque<String> pilha = new ArrayDeque<>();
pilha.push("A");
pilha.push("B");
pilha.push("C");
IO.println("pilha: " + pilha + ", peek " + pilha.peek() + ", pop " + pilha.pop() + ", agora " + pilha);

Deque<String> fila = new ArrayDeque<>();
fila.offer("A");
fila.offer("B");
fila.offer("C");
IO.println("fila: " + fila + ", peek " + fila.peek() + ", poll " + fila.poll() + ", agora " + fila);

### ✏️ Exercício: Inverta as palavras

Uma pilha devolve as coisas na ordem contrária. Empilhe cada palavra da frase e depois desempilhe todas para montar a resposta.

In [ ]:
String reverseWords(String sentence) {
    Deque<String> stack = new ArrayDeque<>();
    // TODO: empilhe cada palavra de sentence.split(" ") e depois desempilhe-as no resultado
    return "";
}

void main() {
    check("três palavras", reverseWords("um dois três"), "três dois um");
    check("uma palavra", reverseWords("oi"), "oi");
    check("palavras repetidas", reverseWords("ser ou não ser"), "ser não ou ser");
}

<details>
<summary>Ver uma resposta</summary>

```java
String reverseWords(String sentence) {
    Deque<String> stack = new ArrayDeque<>();
    for (String word : sentence.split(" ")) {
        stack.push(word);
    }
    StringJoiner result = new StringJoiner(" ");
    while (!stack.isEmpty()) {
        result.add(stack.pop());
    }
    return result.toString();
}

void main() {
    check("três palavras", reverseWords("um dois três"), "três dois um");
    check("uma palavra", reverseWords("oi"), "oi");
    check("palavras repetidas", reverseWords("ser ou não ser"), "ser não ou ser");
}
```

</details>

### ✏️ Exercício: Batata quente

Crianças numeradas de 1 a n ficam em roda e passam uma batata. Depois de cada k passes, quem está com ela sai. Devolva a última criança que sobrar. Uma fila representa a roda: passar é `offer(poll())`.

In [ ]:
int lastChild(int n, int k) {
    Deque<Integer> circle = new ArrayDeque<>();
    for (int i = 1; i <= n; i++) {
        circle.offer(i);
    }
    // TODO: enquanto sobrar mais de uma criança, passe a batata k vezes e tire quem estiver com ela
    return circle.peek();
}

void main() {
    check("uma criança", lastChild(1, 3), 1);
    check("5 crianças, 2 passes", lastChild(5, 2), 4);
    check("6 crianças, 1 passe", lastChild(6, 1), 5);
    check("10 crianças, 4 passes", lastChild(10, 4), 3);
}

<details>
<summary>Ver uma resposta</summary>

```java
int lastChild(int n, int k) {
    Deque<Integer> circle = new ArrayDeque<>();
    for (int i = 1; i <= n; i++) {
        circle.offer(i);
    }
    while (circle.size() > 1) {
        for (int pass = 0; pass < k; pass++) {
            circle.offer(circle.poll());
        }
        circle.poll();
    }
    return circle.peek();
}

void main() {
    check("uma criança", lastChild(1, 3), 1);
    check("5 crianças, 2 passes", lastChild(5, 2), 4);
    check("6 crianças, 1 passe", lastChild(6, 1), 5);
    check("10 crianças, 4 passes", lastChild(10, 4), 3);
}
```

</details>

Dois padrões transformam laços O(n²) em O(n):

- Uma **pilha monotônica** mantém seus valores em ordem decrescente. Quando chega um valor maior, ela desempilha todos os menores, e cada valor desempilhado acabou de achar seu próximo valor maior. Cada valor entra uma vez e sai no máximo uma vez, então o laço dentro do laço custa O(n) no total.
- Um **deque monotônico** faz o mesmo para uma janela deslizante: a frente dele é o máximo da janela, valores que nunca mais podem ser o máximo saem pelo fundo, e valores que saem da janela saem pela frente.

**Uma pilha monotônica, passo a passo:** para cada valor, o valor menor mais próximo à esquerda. Repare na pilha sempre crescente.

In [ ]:
int[] valores = {4, 7, 3, 5, 8, 2, 6};
Deque<Integer> pilha = new ArrayDeque<>();
for (int v : valores) {
    while (!pilha.isEmpty() && pilha.peek() >= v) {
        pilha.pop(); // nunca mais será a resposta de nada à direita de v
    }
    IO.println(v + ": menor mais próximo à esquerda = " + (pilha.isEmpty() ? "nenhum" : pilha.peek()) + "   pilha (topo primeiro) " + pilha);
    pilha.push(v);
}

Duas pilhas, uma para números e outra para operadores, também calculam expressões como `2 + 3 * 4` com a precedência certa.

### ✏️ Exercício: Simplifique um caminho

Num caminho de pastas, `..` sobe uma pasta e `.` fica no lugar. Divida o caminho em `/`, empilhe cada nome de pasta, desempilhe num `..`, e pule `.` e partes vazias. Depois junte o que sobrou.

In [ ]:
String simplify(String path) {
    Deque<String> folders = new ArrayDeque<>();
    // TODO: percorra path.split("/") usando folders como pilha
    return "/";
}

void main() {
    check("sobe uma pasta", simplify("/a/b/../c"), "/a/c");
    check("pontos e barras", simplify("/a/./b//c/"), "/a/b/c");
    check("acima da raiz", simplify("/../x"), "/x");
    check("de volta à raiz", simplify("/a/.."), "/");
}

<details>
<summary>Ver uma resposta</summary>

```java
String simplify(String path) {
    Deque<String> folders = new ArrayDeque<>();
    for (String part : path.split("/")) {
        if (part.equals("..")) {
            folders.pollFirst(); // nada acontece acima da raiz
        } else if (!part.isEmpty() && !part.equals(".")) {
            folders.push(part);
        }
    }
    StringBuilder result = new StringBuilder();
    while (!folders.isEmpty()) {
        result.append('/').append(folders.pollLast()); // o fundo da pilha é a primeira pasta
    }
    return result.length() == 0 ? "/" : result.toString();
}

void main() {
    check("sobe uma pasta", simplify("/a/b/../c"), "/a/c");
    check("pontos e barras", simplify("/a/./b//c/"), "/a/b/c");
    check("acima da raiz", simplify("/../x"), "/x");
    check("de volta à raiz", simplify("/a/.."), "/");
}
```

</details>

Cuidado: `pop()` num `ArrayDeque` vazio lança uma exceção, enquanto `poll()` devolve `null`.

**pop() e poll() num deque vazio:**

In [ ]:
Deque<Integer> vazio = new ArrayDeque<>();
IO.println("poll(): " + vazio.poll());
try {
    vazio.pop();
} catch (NoSuchElementException e) {
    IO.println("pop(): " + e.getClass().getSimpleName());
}

## Leituras

- [Stacks and Queues](https://dev.java/learn/api/collections-and-streams/collections-framework/stacks-queues/) (dev.java (Oracle), 6 min): As tabelas de métodos: quais lançam uma exceção num deque vazio (ou cheio) e quais devolvem null ou false, e por que o ArrayDeque substitui a velha classe Stack.
- [Stacks](https://usaco.guide/gold/stacks) (USACO Guide, 10 min): "Nearest Smaller Element": a pilha guarda só os valores que ainda podem ser resposta, e isso a deixa monotônica. Warmer Days é a mesma ideia com "maior" e para a direita. Escolha Java nas configurações do site para ver código em Java.
- [Sliding Window](https://usaco.guide/gold/sliding-window) (USACO Guide, 15 min): "Sliding Window Maximum in O(N)", Method 1 (Deque): os valores saem pelo fundo quando chega um maior e pela frente quando saem da janela. Pule "With Two Pointers" (a unidade 1 cobriu) e as listas de problemas.

## Verifique

1. Que estrutura um botão Voltar precisa, e qual uma fila de impressão?
2. Por que uma pilha monotônica é O(n), mesmo tendo um laço dentro de outro?
3. Em `8 - 3 - 2`, qual subtração precisa ser feita primeiro, e por quê?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Escalonador round-robin** · Fácil
- **Dias mais quentes** · Médio
- **Pico de carga** · Médio
- **Calculadora** · Difícil